# Deep CFR: Complex-Spec GPU Architecture Screen With Async Exact Evaluation

This experiment tests wider advantage networks on `r4_s4_h2_hp2pt_ss` while keeping the deployable average-strategy networks fixed at `512x512`.

GPU training never waits for exact evaluation. Every five measured training minutes, the learned-average neural policy is saved as an immutable snapshot and submitted to one CPU worker. The worker compiles that policy to a dense table and computes exact exploitability while GPU training continues.

There is no exact generated averaging and no current-strategy exact evaluation in this screen.

In [ ]:
import gc
import json
import os
import sys
import time
from datetime import datetime, timezone
from pathlib import Path

REPO_ROOT = Path.cwd().resolve()
while not (REPO_ROOT / 'liars_poker').is_dir():
    REPO_ROOT = REPO_ROOT.parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch

from liars_poker.algo.deep_cfr import DeepCFRTrainer
from liars_poker.core import GameSpec
from liars_poker.eval.deep_cfr_async import AsyncDeepCFREvaluator
from liars_poker.policies.neural import NeuralMLP, NeuralPolicy, compile_neural_to_dense

assert torch.cuda.is_available(), 'This experiment requires CUDA.'
device = torch.device('cuda')
torch.set_float32_matmul_precision('high')
print('torch:', torch.__version__)
print('GPU:', torch.cuda.get_device_name(0))
print('CPU count:', os.cpu_count())
print('free / total GPU GiB:', tuple(round(x / 2**30, 2) for x in torch.cuda.mem_get_info()))

In [ ]:
spec = GameSpec(
    ranks=4,
    suits=4,
    hand_size=2,
    claim_kinds=('RankHigh', 'Pair', 'TwoPair', 'Trips'),
    suit_symmetry=True,
)

ADVANTAGE_ARCHITECTURES = {
    '1024x1024': (1024, 1024),
    '2048x2048': (2048, 2048),
    '4096x4096': (4096, 4096),
    '2048x4096x2048': (2048, 4096, 2048),
}

STRATEGY_HIDDEN_SIZES = (512, 512)
TRAINING_MINUTES_PER_ARCHITECTURE = 15
SNAPSHOT_EVERY_TRAINING_MINUTES = 5
SEED = 7

TRAVERSALS_PER_PLAYER = 1024
TRAVERSAL_BATCH_SIZE = 1024
FITTING_BATCH_SIZE = 4096
ADVANTAGE_TRAIN_STEPS = 100
STRATEGY_TRAIN_STEPS = 50
ADVANTAGE_POSITIVE_WEIGHT = 0.5

# The evaluator is CPU-only. Restrict its numerical libraries so that the
# background worker leaves CPU capacity for GPU-training orchestration.
EVALUATOR_CPU_THREADS = 16
EVALUATOR_COMPILE_BATCH_SIZE = 65_536
for variable in ('OMP_NUM_THREADS', 'MKL_NUM_THREADS', 'OPENBLAS_NUM_THREADS', 'NUMEXPR_NUM_THREADS'):
    os.environ[variable] = str(EVALUATOR_CPU_THREADS)

run_id = datetime.now(timezone.utc).strftime('%Y%m%d-%H%M%S')
RUN_DIR = REPO_ROOT / 'artifacts' / 'deep_cfr_gpu_architecture_async' / f'{spec.to_short_str()}___{run_id}'
RESULTS_JSONL = RUN_DIR / 'exact_results.jsonl'
RUN_DIR.mkdir(parents=True, exist_ok=True)

print('Spec:', spec.to_short_str())
print('Run directory:', RUN_DIR)
ADVANTAGE_ARCHITECTURES

## Batched compiler sanity check

The optimized compiler should produce the same dense policy regardless of how many infosets are passed through the network at once.

In [ ]:
small_spec = GameSpec(
    ranks=2,
    suits=2,
    hand_size=1,
    claim_kinds=('RankHigh',),
    suit_symmetry=True,
)
small_policy = NeuralPolicy(small_spec, hidden_sizes=(16, 16), device=device)
dense_small_batches = compile_neural_to_dense(small_policy, batch_size=3)
dense_large_batches = compile_neural_to_dense(small_policy, batch_size=4096)
max_strategy_difference = float(np.max(np.abs(dense_small_batches.S - dense_large_batches.S)))
max_reach_difference = max(
    float(np.max(np.abs(dense_small_batches.L_pid0 - dense_large_batches.L_pid0))),
    float(np.max(np.abs(dense_small_batches.L_pid1 - dense_large_batches.L_pid1))),
)
assert max_strategy_difference < 1e-6
assert max_reach_difference < 1e-6
print('max strategy difference:', max_strategy_difference)
print('max reach difference:', max_reach_difference)
del small_policy, dense_small_batches, dense_large_batches
gc.collect()
torch.cuda.empty_cache()

## Experiment helpers

Only the two advantage networks change architecture. Strategy networks remain `512x512`, so learned-average quality remains directly comparable and snapshot size stays small.

In [ ]:
def make_trainer(advantage_hidden_sizes, *, seed):
    trainer = DeepCFRTrainer(
        spec,
        hidden_sizes=STRATEGY_HIDDEN_SIZES,
        device=device,
        seed=seed,
        advantage_buffer_capacity=2_000_000,
        strategy_buffer_capacity=2_000_000,
        learning_rate=1e-3,
        batch_size=FITTING_BATCH_SIZE,
        advantage_train_steps=ADVANTAGE_TRAIN_STEPS,
        strategy_train_steps=STRATEGY_TRAIN_STEPS,
        advantage_positive_weight=ADVANTAGE_POSITIVE_WEIGHT,
        strategy_weighting='linear',
        highest_regret_fallback=True,
        alternating_updates=True,
        validation_fraction=0.02,
        validation_buffer_capacity=20_000,
        traversal_backend='gpu_native',
        traversal_batch_size=TRAVERSAL_BATCH_SIZE,
        device_replay=True,
        fused_optimizer=True,
        amp_dtype=None,
        compile_models=False,
    )

    torch.manual_seed(seed + 10_000)
    trainer.advantage_nets = [
        NeuralMLP(
            trainer.encoder.input_dim,
            trainer.encoder.action_dim,
            advantage_hidden_sizes,
        ).to(device)
        for _ in range(2)
    ]
    trainer.advantage_optimizers = [
        trainer._make_optimizer(model) for model in trainer.advantage_nets
    ]
    trainer._compiled_forwards.clear()
    return trainer


def parameter_count(model):
    return sum(parameter.numel() for parameter in model.parameters())


def append_results(results):
    if not results:
        return
    with RESULTS_JSONL.open('a', encoding='utf-8') as handle:
        for result in results:
            handle.write(json.dumps(result) + '\n')


def load_results():
    if not RESULTS_JSONL.exists():
        return []
    return [json.loads(line) for line in RESULTS_JSONL.read_text(encoding='utf-8').splitlines() if line.strip()]

## Run equal-time GPU training and submit CPU evaluations

The CPU worker may fall behind temporarily. This does not block training; all snapshots are retained under the run directory and the final `wait()` drains the queue after GPU work is complete.

In [ ]:
RUN_SCREEN = True
training_runs = []
queue_history = []

if RUN_SCREEN:
    evaluator = AsyncDeepCFREvaluator(
        RUN_DIR,
        max_workers=1,
        compile_batch_size=EVALUATOR_COMPILE_BATCH_SIZE,
    )
    try:
        for configuration, hidden_sizes in ADVANTAGE_ARCHITECTURES.items():
            print(f'\n=== {configuration} ===')
            trainer = make_trainer(hidden_sizes, seed=SEED)
            training_records = []
            measured_training_s = 0.0
            next_snapshot_s = SNAPSHOT_EVERY_TRAINING_MINUTES * 60.0
            actual_start = time.perf_counter()
            snapshot_index = 0

            while measured_training_s < TRAINING_MINUTES_PER_ARCHITECTURE * 60.0:
                start = time.perf_counter()
                record = trainer.run_iteration(traversals_per_player=TRAVERSALS_PER_PLAYER)
                measured_training_s += time.perf_counter() - start
                record['measured_training_s'] = measured_training_s
                training_records.append(record)

                ready = evaluator.collect_ready()
                append_results(ready)

                if measured_training_s >= next_snapshot_s:
                    validation = trainer.validation_metrics()
                    snapshot_index += 1
                    metadata = {
                        'configuration': configuration,
                        'advantage_hidden_sizes': list(hidden_sizes),
                        'seed': SEED,
                        'measured_training_s': measured_training_s,
                        'snapshot_index': snapshot_index,
                        'advantage_validation_mse': float(np.mean([
                            player['mse'] for player in validation['advantage']
                        ])),
                        'advantage_validation_strategy_tv': float(np.mean([
                            player['strategy_tv'] for player in validation['advantage']
                        ])),
                    }
                    snapshot_path = evaluator.submit(
                        trainer.iteration,
                        trainer.average_policy(),
                        label='learned_average',
                        metadata=metadata,
                    )
                    queue_history.append({
                        'configuration': configuration,
                        'measured_training_s': measured_training_s,
                        'actual_elapsed_s': time.perf_counter() - actual_start,
                        'pending evaluations': evaluator.pending_count,
                    })
                    print(
                        f'[{configuration}] train={measured_training_s / 60:.1f}m '
                        f'iter={trainer.iteration} pending={evaluator.pending_count} '
                        f'snapshot={snapshot_path.name}'
                    )
                    while next_snapshot_s <= measured_training_s:
                        next_snapshot_s += SNAPSHOT_EVERY_TRAINING_MINUTES * 60.0

            training_runs.append({
                'configuration': configuration,
                'hidden_sizes': tuple(hidden_sizes),
                'advantage_parameters': parameter_count(trainer.advantage_nets[0]),
                'strategy_parameters': parameter_count(trainer.strategy_nets[0]),
                'iterations': trainer.iteration,
                'measured_training_s': measured_training_s,
                'actual_training_wall_s': time.perf_counter() - actual_start,
                'training_records': training_records,
            })
            del trainer
            gc.collect()
            torch.cuda.empty_cache()

        print(f'\nGPU screen complete; draining {evaluator.pending_count} pending evaluations...')
        append_results(evaluator.wait())
    finally:
        evaluator.close(wait=True)

print('Completed exact evaluations:', len(load_results()))
print('Results:', RESULTS_JSONL)

## Results

The x-axis is measured GPU-training time attached to each immutable snapshot, not evaluation completion time. Exact evaluation latency therefore cannot distort the learning curves.

In [ ]:
exact_results = load_results()
exact_df = pd.DataFrame(exact_results)

training_summary_rows = []
for run in training_runs:
    records = run['training_records']
    final = records[-1]
    training_summary_rows.append({
        'configuration': run['configuration'],
        'advantage parameters': run['advantage_parameters'],
        'strategy parameters': run['strategy_parameters'],
        'iterations completed': run['iterations'],
        'measured training min': run['measured_training_s'] / 60.0,
        'actual training wall min': run['actual_training_wall_s'] / 60.0,
        'mean traversal s': float(np.mean([row['timing']['traversal_s'] for row in records])),
        'mean advantage fit s': float(np.mean([row['timing']['advantage_training_s'] for row in records])),
        'mean strategy fit s': float(np.mean([row['timing']['strategy_training_s'] for row in records])),
        'advantage records seen': sum(final['advantage_records_seen']),
        'strategy records seen': sum(final['strategy_records_seen']),
    })

training_summary_df = pd.DataFrame(training_summary_rows).set_index('configuration')
display(training_summary_df.style.format(precision=6))

if not exact_df.empty:
    exact_df = exact_df.sort_values(['configuration', 'measured_training_s'])
    display(exact_df[[
        'configuration', 'iter', 'measured_training_s', 'exploitability',
        'dense_compile_s', 'exact_br_s', 'evaluation_s',
        'advantage_validation_mse', 'advantage_validation_strategy_tv',
    ]])

In [ ]:
def normalized_auc(x, y):
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)
    if len(x) < 2 or x[-1] <= x[0]:
        return np.nan
    return float(np.trapezoid(y, x) / (x[-1] - x[0]))


summary_rows = []
for configuration, group in exact_df.groupby('configuration'):
    group = group.sort_values('measured_training_s')
    summary_rows.append({
        'configuration': configuration,
        'final learned-average exploitability': group['exploitability'].iloc[-1],
        'best learned-average exploitability': group['exploitability'].min(),
        'learned-average normalized AUC': normalized_auc(
            group['measured_training_s'], group['exploitability']
        ),
        'final held-out advantage MSE': group['advantage_validation_mse'].iloc[-1],
        'final held-out strategy TV': group['advantage_validation_strategy_tv'].iloc[-1],
        'mean dense compile s': group['dense_compile_s'].mean(),
        'mean exact BR s': group['exact_br_s'].mean(),
        'mean total evaluation s': group['evaluation_s'].mean(),
    })

architecture_summary_df = pd.DataFrame(summary_rows).set_index('configuration').sort_values(
    ['learned-average normalized AUC', 'final learned-average exploitability']
)
display(architecture_summary_df.style.format(precision=6).background_gradient(
    subset=[
        'final learned-average exploitability',
        'best learned-average exploitability',
        'learned-average normalized AUC',
    ],
    cmap='RdYlGn_r',
))

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(15, 10))

for configuration, group in exact_df.groupby('configuration', sort=False):
    group = group.sort_values('measured_training_s')
    x = group['measured_training_s'] / 60.0
    axes[0, 0].plot(x, group['exploitability'], marker='o', label=configuration)
    axes[0, 1].plot(x, group['advantage_validation_mse'], marker='o', label=configuration)
    axes[1, 0].plot(x, group['advantage_validation_strategy_tv'], marker='o', label=configuration)

queue_df = pd.DataFrame(queue_history)
if not queue_df.empty:
    for configuration, group in queue_df.groupby('configuration', sort=False):
        axes[1, 1].plot(
            group['measured_training_s'] / 60.0,
            group['pending evaluations'],
            marker='o',
            label=configuration,
        )

plot_specs = [
    ('Learned-average exact exploitability', 'Exploitability'),
    ('Held-out advantage MSE', 'MSE'),
    ('Held-out induced-strategy TV', 'TV'),
    ('CPU evaluation queue at snapshot submission', 'Pending evaluations'),
]
for ax, (title, ylabel) in zip(axes.flat, plot_specs):
    ax.set(title=title, xlabel='Measured GPU-training minutes', ylabel=ylabel)
    ax.grid(True, alpha=0.3)
    ax.legend(fontsize=8)
axes[0, 0].set_yscale('log')
fig.tight_layout();

## Follow-up: fitting batch size for `4096x4096`

The first screen showed that `4096x4096` crossed a fitting-cost boundary. This follow-up holds architecture, optimizer steps, traversal budget, seed, and measured GPU-training time fixed while comparing fitting batches `4096`, `8192`, and `16384`.

Larger batches change two things at once: GPU utilization and the number of replay examples consumed by each optimizer step. That is intentional here because the practical question is which setting produces the best learned average per wall-clock training minute.

In [ ]:
BATCH_SCREEN_ARCHITECTURE = (4096, 4096)
BATCH_SCREEN_SIZES = (4096, 8192, 16384)
BATCH_SCREEN_MINUTES_PER_CONFIGURATION = 15
BATCH_SCREEN_SNAPSHOT_EVERY_MINUTES = 5
BATCH_SCREEN_SEED = 17

batch_run_id = datetime.now(timezone.utc).strftime('%Y%m%d-%H%M%S')
BATCH_RUN_DIR = (
    REPO_ROOT / 'artifacts' / 'deep_cfr_gpu_batch_screen'
    / f'{spec.to_short_str()}___{batch_run_id}'
)
BATCH_RESULTS_JSONL = BATCH_RUN_DIR / 'exact_results.jsonl'
BATCH_RUN_DIR.mkdir(parents=True, exist_ok=True)

print('Batch-screen run directory:', BATCH_RUN_DIR)
BATCH_SCREEN_SIZES

In [ ]:
def make_batch_screen_trainer(batch_size, *, seed):
    trainer = DeepCFRTrainer(
        spec,
        hidden_sizes=STRATEGY_HIDDEN_SIZES,
        device=device,
        seed=seed,
        advantage_buffer_capacity=2_000_000,
        strategy_buffer_capacity=2_000_000,
        learning_rate=1e-3,
        batch_size=batch_size,
        advantage_train_steps=ADVANTAGE_TRAIN_STEPS,
        strategy_train_steps=STRATEGY_TRAIN_STEPS,
        advantage_positive_weight=ADVANTAGE_POSITIVE_WEIGHT,
        strategy_weighting='linear',
        highest_regret_fallback=True,
        alternating_updates=True,
        validation_fraction=0.02,
        validation_buffer_capacity=20_000,
        traversal_backend='gpu_native',
        traversal_batch_size=TRAVERSAL_BATCH_SIZE,
        device_replay=True,
        fused_optimizer=True,
        amp_dtype=None,
        compile_models=False,
    )

    torch.manual_seed(seed + 20_000)
    trainer.advantage_nets = [
        NeuralMLP(
            trainer.encoder.input_dim,
            trainer.encoder.action_dim,
            BATCH_SCREEN_ARCHITECTURE,
        ).to(device)
        for _ in range(2)
    ]
    trainer.advantage_optimizers = [
        trainer._make_optimizer(model) for model in trainer.advantage_nets
    ]
    trainer._compiled_forwards.clear()
    return trainer


def append_batch_results(results):
    if not results:
        return
    with BATCH_RESULTS_JSONL.open('a', encoding='utf-8') as handle:
        for result in results:
            handle.write(json.dumps(result) + '\n')


def load_batch_results():
    if not BATCH_RESULTS_JSONL.exists():
        return []
    return [
        json.loads(line)
        for line in BATCH_RESULTS_JSONL.read_text(encoding='utf-8').splitlines()
        if line.strip()
    ]

In [ ]:
RUN_BATCH_SCREEN = True
batch_training_runs = []
batch_queue_history = []

if RUN_BATCH_SCREEN:
    batch_evaluator = AsyncDeepCFREvaluator(
        BATCH_RUN_DIR,
        max_workers=1,
        compile_batch_size=EVALUATOR_COMPILE_BATCH_SIZE,
    )
    try:
        for batch_size in BATCH_SCREEN_SIZES:
            configuration = f'4096x4096; batch={batch_size}'
            print(f'\n=== {configuration} ===')
            trainer = make_batch_screen_trainer(batch_size, seed=BATCH_SCREEN_SEED)
            training_records = []
            measured_training_s = 0.0
            next_snapshot_s = BATCH_SCREEN_SNAPSHOT_EVERY_MINUTES * 60.0
            actual_start = time.perf_counter()
            snapshot_index = 0
            torch.cuda.reset_peak_memory_stats()

            while measured_training_s < BATCH_SCREEN_MINUTES_PER_CONFIGURATION * 60.0:
                start = time.perf_counter()
                record = trainer.run_iteration(traversals_per_player=TRAVERSALS_PER_PLAYER)
                measured_training_s += time.perf_counter() - start
                record['measured_training_s'] = measured_training_s
                training_records.append(record)

                append_batch_results(batch_evaluator.collect_ready())

                if measured_training_s >= next_snapshot_s:
                    validation = trainer.validation_metrics()
                    snapshot_index += 1
                    metadata = {
                        'configuration': configuration,
                        'batch_size': batch_size,
                        'advantage_hidden_sizes': list(BATCH_SCREEN_ARCHITECTURE),
                        'seed': BATCH_SCREEN_SEED,
                        'measured_training_s': measured_training_s,
                        'snapshot_index': snapshot_index,
                        'advantage_validation_mse': float(np.mean([
                            player['mse'] for player in validation['advantage']
                        ])),
                        'advantage_validation_strategy_tv': float(np.mean([
                            player['strategy_tv'] for player in validation['advantage']
                        ])),
                    }
                    batch_evaluator.submit(
                        trainer.iteration,
                        trainer.average_policy(),
                        label='learned_average',
                        metadata=metadata,
                    )
                    batch_queue_history.append({
                        'configuration': configuration,
                        'measured_training_s': measured_training_s,
                        'pending evaluations': batch_evaluator.pending_count,
                    })
                    print(
                        f'[{configuration}] train={measured_training_s / 60:.1f}m '
                        f'iter={trainer.iteration} pending={batch_evaluator.pending_count}'
                    )
                    while next_snapshot_s <= measured_training_s:
                        next_snapshot_s += BATCH_SCREEN_SNAPSHOT_EVERY_MINUTES * 60.0

            batch_training_runs.append({
                'configuration': configuration,
                'batch_size': batch_size,
                'iterations': trainer.iteration,
                'measured_training_s': measured_training_s,
                'actual_training_wall_s': time.perf_counter() - actual_start,
                'peak_GPU_GiB': torch.cuda.max_memory_allocated() / 2**30,
                'training_records': training_records,
            })
            del trainer
            gc.collect()
            torch.cuda.empty_cache()

        print(f'\nGPU batch screen complete; draining {batch_evaluator.pending_count} evaluations...')
        append_batch_results(batch_evaluator.wait())
    finally:
        batch_evaluator.close(wait=True)

print('Completed exact evaluations:', len(load_batch_results()))
print('Results:', BATCH_RESULTS_JSONL)

In [ ]:
batch_exact_df = pd.DataFrame(load_batch_results())
if not batch_exact_df.empty:
    batch_exact_df = batch_exact_df.sort_values(['batch_size', 'measured_training_s'])

batch_summary_rows = []
for run in batch_training_runs:
    records = run['training_records']
    mean_advantage_fit_s = float(np.mean([
        row['timing']['advantage_training_s'] for row in records
    ]))
    mean_strategy_fit_s = float(np.mean([
        row['timing']['strategy_training_s'] for row in records
    ]))
    group = batch_exact_df[batch_exact_df['batch_size'] == run['batch_size']]
    group = group.sort_values('measured_training_s')
    batch_summary_rows.append({
        'batch size': run['batch_size'],
        'iterations completed': run['iterations'],
        'measured training min': run['measured_training_s'] / 60.0,
        'actual training wall min': run['actual_training_wall_s'] / 60.0,
        'peak GPU GiB': run['peak_GPU_GiB'],
        'final learned-average exploitability': group['exploitability'].iloc[-1],
        'best learned-average exploitability': group['exploitability'].min(),
        'learned-average normalized AUC': normalized_auc(
            group['measured_training_s'], group['exploitability']
        ),
        'mean traversal s': float(np.mean([
            row['timing']['traversal_s'] for row in records
        ])),
        'mean advantage fit s': mean_advantage_fit_s,
        'mean strategy fit s': mean_strategy_fit_s,
        'advantage samples / fit s': (
            2 * ADVANTAGE_TRAIN_STEPS * run['batch_size'] / mean_advantage_fit_s
        ),
        'strategy samples / fit s': (
            2 * STRATEGY_TRAIN_STEPS * run['batch_size'] / mean_strategy_fit_s
        ),
        'final held-out advantage MSE': group['advantage_validation_mse'].iloc[-1],
        'final held-out strategy TV': group['advantage_validation_strategy_tv'].iloc[-1],
    })

batch_summary_df = pd.DataFrame(batch_summary_rows).set_index('batch size').sort_index()
display(batch_summary_df.style.format(precision=6).background_gradient(
    subset=[
        'final learned-average exploitability',
        'best learned-average exploitability',
        'learned-average normalized AUC',
    ],
    cmap='RdYlGn_r',
))

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(15, 10))

for batch_size, group in batch_exact_df.groupby('batch_size'):
    group = group.sort_values('measured_training_s')
    label = f'batch={batch_size}'
    x = group['measured_training_s'] / 60.0
    axes[0, 0].plot(x, group['exploitability'], marker='o', label=label)
    axes[0, 1].plot(x, group['advantage_validation_mse'], marker='o', label=label)
    axes[1, 0].plot(x, group['advantage_validation_strategy_tv'], marker='o', label=label)

flat_batch_summary = batch_summary_df.reset_index()
axes[1, 1].plot(
    flat_batch_summary['batch size'],
    flat_batch_summary['mean advantage fit s'],
    marker='o',
    label='advantage fit seconds',
)
throughput_axis = axes[1, 1].twinx()
throughput_axis.plot(
    flat_batch_summary['batch size'],
    flat_batch_summary['advantage samples / fit s'],
    marker='s',
    color='C3',
    label='advantage samples / second',
)

for ax, title, ylabel in [
    (axes[0, 0], 'Learned-average exact exploitability', 'Exploitability'),
    (axes[0, 1], 'Held-out advantage MSE', 'MSE'),
    (axes[1, 0], 'Held-out induced-strategy TV', 'TV'),
]:
    ax.set(title=title, xlabel='Measured GPU-training minutes', ylabel=ylabel)
    ax.grid(True, alpha=0.3)
    ax.legend()
axes[0, 0].set_yscale('log')
axes[1, 1].set(
    title='Fitting cost and throughput',
    xlabel='Fitting batch size',
    ylabel='Advantage fitting seconds per iteration',
)
throughput_axis.set_ylabel('Advantage samples per fitting second')
axes[1, 1].grid(True, alpha=0.3)
lines1, labels1 = axes[1, 1].get_legend_handles_labels()
lines2, labels2 = throughput_axis.get_legend_handles_labels()
axes[1, 1].legend(lines1 + lines2, labels1 + labels2, loc='best')
fig.tight_layout();